# Notebook 03: Feature Engineering

## Objective
This notebook focuses on transforming cleaned data into business-ready features required for logistics analysis.

### Key Feature Engineering Steps:
- Create Factory column using product-to-factory mapping
- Define route structures (Factory → State, Factory → Region)
- Create time-based features
- Classify delay status
- Categorize shipping speed

These features will enable route-level analysis and efficiency benchmarking in the next steps.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# Load cleaned dataset
file_path = r"../data/cleaned_nassau_data.csv"
ncd = pd.read_csv(file_path)

ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7


## Factory Mapping

Since the dataset does not contain a Factory column, we will create it using the provided product-to-factory mapping.

In [3]:
# Factory mapping dictionary
factory_mapping = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",

    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",

    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",

    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",

    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory"
}

In [4]:
# Create Factory column
ncd["Factory"] = ncd["Product Name"].map(factory_mapping)

ncd[["Product Name", "Factory"]].head(10)

,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
5,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
6,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
7,Wonka Bar - Milk Chocolate,Wicked Choccy's
8,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
9,Wonka Bar - Milk Chocolate,Wicked Choccy's


In [5]:
# Check missing factory mappings
ncd["Factory"].isnull().sum()

np.int64(0)

## Route Creation

Define logistics routes based on:
- Factory → Customer State
- Factory → Customer Region

In [6]:
# Route to State
ncd["Route_State"] = ncd["Factory"] + " → " + ncd["State/Province"]

# Route to Region
ncd["Route_Region"] = ncd["Factory"] + " → " + ncd["Region"]

ncd[["Factory", "State/Province", "Route_State"]].head()

,Factory,State/Province,Route_State
0,Wicked Choccy's,Texas,Wicked Choccy's → Texas
1,Wicked Choccy's,Illinois,Wicked Choccy's → Illinois
2,Lot's O' Nuts,Illinois,Lot's O' Nuts → Illinois
3,Lot's O' Nuts,Illinois,Lot's O' Nuts → Illinois
4,Wicked Choccy's,Pennsylvania,Wicked Choccy's → Pennsylvania


## Time-based Features

Extract additional time features for trend and seasonal analysis.

In [7]:
# Convert to datetime (safety check)
ncd["Order Date"] = pd.to_datetime(ncd["Order Date"])

# Extract features
ncd["Order Month"] = ncd["Order Date"].dt.month
ncd["Order Year"] = ncd["Order Date"].dt.year
ncd["Order Month Name"] = ncd["Order Date"].dt.month_name()

ncd[["Order Date", "Order Month", "Order Month Name"]].head()

,Order Date,Order Month,Order Month Name
0,2024-01-03,1,January
1,2024-01-04,1,January
2,2024-01-04,1,January
3,2024-01-04,1,January
4,2024-01-05,1,January


## Delay Classification

Define delayed shipments based on lead time threshold.

In [8]:
# Define delay threshold (business logic)
delay_threshold = 5

ncd["Delay Status"] = np.where(
    ncd["Shipping Lead Time"] > delay_threshold,
    "Delayed",
    "On-Time"
)

ncd[["Shipping Lead Time", "Delay Status"]].head()

,Shipping Lead Time,Delay Status
0,6,Delayed
1,7,Delayed
2,5,On-Time
3,7,Delayed
4,7,Delayed


In [9]:
# Check delay distribution
ncd["Delay Status"].value_counts()

Delay Status
On-Time    7738
Delayed    2456
Name: count, dtype: int64

## Shipping Speed Category

Categorize shipping modes into speed levels for comparison.

In [10]:
# Shipping speed mapping
speed_mapping = {
    "Standard Class": "Slow",
    "Second Class": "Medium",
    "First Class": "Fast",
    "Same Day": "Fastest"
}

ncd["Shipping Speed Category"] = ncd["Ship Mode"].map(speed_mapping)

ncd[["Ship Mode", "Shipping Speed Category"]].head()

,Ship Mode,Shipping Speed Category
0,Standard Class,Slow
1,Standard Class,Slow
2,Standard Class,Slow
3,Standard Class,Slow
4,Standard Class,Slow


In [11]:
ncd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Row ID                   10194 non-null  int64         
 1   Order ID                 10194 non-null  object        
 2   Order Date               10194 non-null  datetime64[ns]
 3   Ship Date                10194 non-null  object        
 4   Ship Mode                10194 non-null  object        
 5   Customer ID              10194 non-null  int64         
 6   Country/Region           10194 non-null  object        
 7   City                     10194 non-null  object        
 8   State/Province           10194 non-null  object        
 9   Postal Code              10194 non-null  object        
 10  Division                 10194 non-null  object        
 11  Region                   10194 non-null  object        
 12  Product ID               10194 n

In [12]:
ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time,Factory,Route_State,Route_Region,Order Month,Order Year,Order Month Name,Delay Status,Shipping Speed Category
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6,Wicked Choccy's,Wicked Choccy's → Texas,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7,Wicked Choccy's,Wicked Choccy's → Illinois,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,On-Time,Slow
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,Delayed,Slow
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7,Wicked Choccy's,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic,1,2024,January,Delayed,Slow


In [13]:
# Save updated dataset
output_path = r"../data/featured_nassau_data.csv"
ncd.to_csv(output_path, index=False)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
